In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import os
import sys

In [2]:
# Get current file location
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().parents[1]

# add src to python path
sys.path.append(str(BASE_DIR))

# Path to data folder
DATA_DIR = BASE_DIR / "data"

# load data
data_train = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None)
data_test = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None)
data_rul = pd.read_csv(DATA_DIR / "RUL_FD001.txt", header=None)

# prepare name for each columns
cols = (
    ['engine_id','cycle'] +
    [f'setting_{i}' for i in range(1,4)] +
    [f'sensor_{i}' for i in range(1,22)]
)

In [3]:
data_train.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [4]:
from src.data.preprocess import preprocess

train_pre, test_pre = preprocess(data_train, data_test, cols, clip=False)

In [5]:
train_pre.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191.0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190.0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189.0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188.0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187.0


In [6]:
from src.features.build_features import build_features

train_fe, test_fe = build_features(train_pre, test_pre)

sensors with low variance:
sensor_1     0.000000e+00
sensor_5     3.155597e-30
sensor_6     1.929279e-06
sensor_10    0.000000e+00
sensor_16    1.926023e-34
sensor_18    0.000000e+00
sensor_19    0.000000e+00
dtype: float64


In [7]:
train_fe.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_diff,sensor_9_diff,sensor_11_diff,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_17_diff,sensor_20_diff,sensor_21_diff
0,1,1,-0.0007,-0.0004,100.0,-1.721725,-0.134255,-0.925936,1.121141,-0.516338,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,0.0019,-0.0003,100.0,-1.061780,0.211528,-0.643726,0.431930,-0.798093,...,-0.281755,-0.096004,0.074884,0.840637,0.695244,-0.373774,0.327964,0.000000,-0.331965,0.042495
2,1,3,-0.0043,0.0003,100.0,-0.661813,-0.413166,-0.525953,1.008155,-0.234584,...,0.563509,0.401678,-0.823720,0.189821,-0.556195,0.091215,-0.373292,-1.291384,-0.276637,-0.733499
3,1,4,0.0007,0.0000,100.0,-0.661813,-1.261314,-0.784831,1.222827,0.188048,...,0.422632,-0.156686,-0.524186,0.596581,0.695244,0.031454,-1.322521,1.291384,-0.387292,0.274369
4,1,5,-0.0019,-0.0002,100.0,-0.621816,-1.251528,-0.301518,0.714393,-0.516338,...,-0.704386,0.256766,0.561628,-0.908431,-0.556195,-0.001573,1.631820,0.645692,0.110655,0.281760


In [8]:
# train-test-split
X = train_fe.drop(columns=["RUL"])
y = train_fe["RUL"]

groups = train_fe["engine_id"]

split = GroupShuffleSplit(test_size=0.2, n_splits=1)

train_idx, val_idx = next(split.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# scaled data for NN-based model
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [9]:
from src.features.build_features import create_sequences

# Reshape into sequences
X_seq, y_seq = create_sequences(train_fe)

In [14]:
X_train_scaled.shape

(16475, 61)

In [16]:
X_seq.shape

(17631, 30, 56)